In [1]:
import numpy as np
from numpy.linalg import eigvalsh
from scipy.linalg import expm

In [2]:
'''
Core math utilities
'''
def partial_trace(rho, keep, dims):
    n = len(dims)
    rho_tensor = rho.reshape(dims + dims)
    trace_out = [i for i in range(n) if i not in keep]
    for sys in sorted(trace_out, reverse=True):
        n_current = len(rho_tensor.shape) // 2
        rho_tensor = np.trace(rho_tensor, axis1=sys, axis2=sys + n_current)
    d_keep = int(np.prod([dims[i] for i in keep]))
    return rho_tensor.reshape(d_keep, d_keep)

def von_neumann_entropy(rho, base=2, tol=1e-12):
    eigvals = eigvalsh(rho)
    eigvals = eigvals[eigvals > tol]
    if len(eigvals) == 0:
        return 0.0
    return float(-np.sum(eigvals * np.log(eigvals) / np.log(base)))

def jordan_product(A, B):
    return 0.5 * (A @ B + B @ A)

def jordan_associator(A, B, C):
    return jordan_product(jordan_product(A, B), C) - jordan_product(A, jordan_product(B, C))

def commutator(A, B):
    return A @ B - B @ A

def to_real_scalar(x, tol=1e-10):
    x = np.real_if_close(x, tol=1000)
    if np.iscomplexobj(x):
        if abs(np.imag(x)) < tol:
            return float(np.real(x))
        raise ValueError(f"Expected nearly real scalar, got {x}")
    return float(x)

def safe_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 3:
        return np.nan
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])

In [3]:
'''
Basic operators and tensor helpers
'''
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
ket_plus = (ket0 + ket1) / np.sqrt(2)

def kron_all(op_list):
    out = np.array([[1.0 + 0j]])
    for op in op_list:
        out = np.kron(out, op)
    return out

def embed_two_body(n_qubits, i, j, op_i, op_j):
    ops = [I2] * n_qubits
    ops[i] = op_i
    ops[j] = op_j
    return kron_all(ops)

def xyz_edge_hamiltonian(n_qubits, i, j, Jx, Jy, Jz):
    return (
        Jx * embed_two_body(n_qubits, i, j, sx, sx)
        + Jy * embed_two_body(n_qubits, i, j, sy, sy)
        + Jz * embed_two_body(n_qubits, i, j, sz, sz)
    )

In [4]:
'''
Initial state generators
'''
def haar_random_state(dim, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    z = rng.normal(size=dim) + 1j * rng.normal(size=dim)
    z = z / np.linalg.norm(z)
    return z.astype(complex)

def random_single_qubit_state(rng=None):
    if rng is None:
        rng = np.random.default_rng()
    z = rng.normal(size=2) + 1j * rng.normal(size=2)
    z = z / np.linalg.norm(z)
    return z.astype(complex)

def make_initial_state(init_mode="plus", rng=None, custom_state=None):
    """
    Return a normalized 4-qubit pure state vector (length 16).

    init_mode:
      - "plus"           : |+ + + 0>
      - "random_product" : |n1> ⊗ |n2> ⊗ |n3> ⊗ |0>
      - "haar"           : Haar-random 4-qubit pure state
      - "custom"         : user-supplied custom_state
    """
    if rng is None:
        rng = np.random.default_rng()

    if init_mode == "plus":
        psi0 = kron_all([ket_plus, ket_plus, ket_plus, ket0]).reshape(-1)

    elif init_mode == "random_product":
        psi_A = random_single_qubit_state(rng)
        psi_B = random_single_qubit_state(rng)
        psi_C = random_single_qubit_state(rng)
        psi0 = kron_all([psi_A, psi_B, psi_C, ket0]).reshape(-1)

    elif init_mode == "haar":
        psi0 = haar_random_state(16, rng=rng)

    elif init_mode == "custom":
        if custom_state is None:
            raise ValueError("custom_state must be provided when init_mode='custom'")
        psi0 = np.asarray(custom_state, dtype=complex).reshape(-1)
        psi0 = psi0 / np.linalg.norm(psi0)

    else:
        raise ValueError(f"Unknown init_mode: {init_mode}")

    return psi0

In [5]:
'''
Cell5: State observables
'''
def mutual_info_2q(rho_2q):
    rho_1 = partial_trace(rho_2q, keep=[0], dims=[2, 2])
    rho_2 = partial_trace(rho_2q, keep=[1], dims=[2, 2])
    return (
        von_neumann_entropy(rho_1)
        + von_neumann_entropy(rho_2)
        - von_neumann_entropy(rho_2q)
    )

def I3_of_rhoABC(rho_ABC):
    rho_A  = partial_trace(rho_ABC, keep=[0], dims=[2, 2, 2])
    rho_B  = partial_trace(rho_ABC, keep=[1], dims=[2, 2, 2])
    rho_C  = partial_trace(rho_ABC, keep=[2], dims=[2, 2, 2])
    rho_AB = partial_trace(rho_ABC, keep=[0, 1], dims=[2, 2, 2])
    rho_AC = partial_trace(rho_ABC, keep=[0, 2], dims=[2, 2, 2])
    rho_BC = partial_trace(rho_ABC, keep=[1, 2], dims=[2, 2, 2])

    I3 = (
        von_neumann_entropy(rho_A)
        + von_neumann_entropy(rho_B)
        + von_neumann_entropy(rho_C)
        - von_neumann_entropy(rho_AB)
        - von_neumann_entropy(rho_AC)
        - von_neumann_entropy(rho_BC)
        + von_neumann_entropy(rho_ABC)
    )
    return to_real_scalar(I3)

def compute_state_observables(rho_ABC):
    rho_AB = partial_trace(rho_ABC, keep=[0, 1], dims=[2, 2, 2])
    rho_AC = partial_trace(rho_ABC, keep=[0, 2], dims=[2, 2, 2])
    rho_BC = partial_trace(rho_ABC, keep=[1, 2], dims=[2, 2, 2])

    I_AB = mutual_info_2q(rho_AB)
    I_AC = mutual_info_2q(rho_AC)
    I_BC = mutual_info_2q(rho_BC)
    P_pair = I_AB + I_AC + I_BC

    I3 = I3_of_rhoABC(rho_ABC)
    I3_minus = max(0.0, -I3)

    S_ABC = von_neumann_entropy(rho_ABC)
    eigvals = np.sort(np.real_if_close(np.linalg.eigvalsh(rho_ABC)))[::-1]

    return {
        "rho_AB": rho_AB,
        "rho_AC": rho_AC,
        "rho_BC": rho_BC,
        "I_AB": I_AB,
        "I_AC": I_AC,
        "I_BC": I_BC,
        "P_pair": P_pair,
        "I3": I3,
        "I3_minus": I3_minus,
        "S_ABC": S_ABC,
        "eigvals": eigvals,
    }

In [6]:
'''
Cell 6：Mediator / probe observables
'''
def mediator_metrics(rho_ABC, H_AB_3, H_BC_3, H_AC_3):
    # Middle operator is the mediator
    A_AB = jordan_associator(H_BC_3, H_AB_3, H_AC_3)  # mediator = H_AB
    A_BC = jordan_associator(H_AB_3, H_BC_3, H_AC_3)  # mediator = H_BC
    A_AC = jordan_associator(H_AB_3, H_AC_3, H_BC_3)  # mediator = H_AC

    j_AB = np.linalg.norm(A_AB, 'fro')
    j_BC = np.linalg.norm(A_BC, 'fro')
    j_AC = np.linalg.norm(A_AC, 'fro')
    J_tot = np.sqrt(j_AB**2 + j_BC**2 + j_AC**2)

    m_AB = to_real_scalar(np.trace(rho_ABC @ A_AB))
    m_BC = to_real_scalar(np.trace(rho_ABC @ A_BC))
    m_AC = to_real_scalar(np.trace(rho_ABC @ A_AC))
    J_rho_exp = np.sqrt(np.abs(m_AB)**2 + np.abs(m_BC)**2 + np.abs(m_AC)**2)

    eta = J_rho_exp / J_tot if J_tot > 1e-12 else np.nan

    return {
        "A_AB": A_AB,
        "A_BC": A_BC,
        "A_AC": A_AC,
        "j_AB": j_AB,
        "j_BC": j_BC,
        "j_AC": j_AC,
        "J_tot": J_tot,
        "m_AB": m_AB,
        "m_BC": m_BC,
        "m_AC": m_AC,
        "J_rho_exp": J_rho_exp,
        "eta": eta,
    }

def compute_probe_observables(rho_ABC, H_AB_3, H_BC_3, H_AC_3):
    return mediator_metrics(rho_ABC, H_AB_3, H_BC_3, H_AC_3)

In [7]:
'''
Cell 7：Family presets and coefficient builder
'''
R = np.sqrt(3.0)

shape_AB = np.array([1.00, 0.70, 0.30], dtype=float)
shape_BC = np.array([0.80, 1.00, 0.50], dtype=float)
shape_AC = np.array([0.60, 0.40, 1.00], dtype=float)
shape_CD = np.array([1.00, 0.80, 0.60], dtype=float)

def unit_direction(theta, phi):
    return np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta)
    ], dtype=float)

def normalized_edge_coeffs(theta, phi, shape, R=np.sqrt(3.0), eps=1e-12):
    vec = unit_direction(theta, phi) * shape
    norm = np.linalg.norm(vec)
    if norm < eps:
        raise ValueError("Degenerate coefficient vector; choose different theta/phi.")
    return R * vec / norm

def build_edge_coefficients(theta, phi, gD=0.20, R=np.sqrt(3.0)):
    J_AB = normalized_edge_coeffs(theta, phi, shape_AB, R=R)
    J_BC = normalized_edge_coeffs(theta, phi, shape_BC, R=R)
    J_AC = normalized_edge_coeffs(theta, phi, shape_AC, R=R)
    J_CD = normalized_edge_coeffs(theta, phi, shape_CD, R=gD * R)
    return {
        "J_AB": J_AB,
        "J_BC": J_BC,
        "J_AC": J_AC,
        "J_CD": J_CD,
    }

In [8]:
'''
Cell 8：Hamiltonian builders + evolution
'''
def build_hamiltonians_4q(J_AB, J_BC, J_AC, J_CD):
    # qubit order on ABCD = A(0), B(1), C(2), D(3)
    H_AB_4 = xyz_edge_hamiltonian(4, 0, 1, *J_AB)
    H_BC_4 = xyz_edge_hamiltonian(4, 1, 2, *J_BC)
    H_AC_4 = xyz_edge_hamiltonian(4, 0, 2, *J_AC)
    H_CD_4 = xyz_edge_hamiltonian(4, 2, 3, *J_CD)
    H_tot_4 = H_AB_4 + H_BC_4 + H_AC_4 + H_CD_4
    return {
        "H_AB_4": H_AB_4,
        "H_BC_4": H_BC_4,
        "H_AC_4": H_AC_4,
        "H_CD_4": H_CD_4,
        "H_tot_4": H_tot_4,
    }

def build_hamiltonians_3q(J_AB, J_BC, J_AC):
    # qubit order on ABC = A(0), B(1), C(2)
    H_AB_3 = xyz_edge_hamiltonian(3, 0, 1, *J_AB)
    H_BC_3 = xyz_edge_hamiltonian(3, 1, 2, *J_BC)
    H_AC_3 = xyz_edge_hamiltonian(3, 0, 2, *J_AC)
    return {
        "H_AB_3": H_AB_3,
        "H_BC_3": H_BC_3,
        "H_AC_3": H_AC_3,
    }

def evolve_to_rhoABC(psi0, H_tot_4, t):
    U = expm(-1j * t * H_tot_4)
    psi_t = U @ psi0
    rho4 = np.outer(psi_t, psi_t.conj())
    rho_ABC = partial_trace(rho4, keep=[0, 1, 2], dims=[2, 2, 2, 2])
    return rho_ABC, psi_t

In [9]:
'''
Cell 9：Main orchestrator
'''
def run_fixed_norm_case(
    theta,
    phi,
    gD=0.20,
    t=0.10,
    init_mode="plus",
    rng=None,
    custom_state=None,
    verbose=True,
):
    psi0 = make_initial_state(init_mode=init_mode, rng=rng, custom_state=custom_state)

    coeffs = build_edge_coefficients(theta, phi, gD=gD, R=R)
    J_AB, J_BC, J_AC, J_CD = coeffs["J_AB"], coeffs["J_BC"], coeffs["J_AC"], coeffs["J_CD"]

    H4 = build_hamiltonians_4q(J_AB, J_BC, J_AC, J_CD)
    H3 = build_hamiltonians_3q(J_AB, J_BC, J_AC)

    rho_ABC, psi_t = evolve_to_rhoABC(psi0, H4["H_tot_4"], t)

    obs_state = compute_state_observables(rho_ABC)
    obs_probe = compute_probe_observables(rho_ABC, H3["H_AB_3"], H3["H_BC_3"], H3["H_AC_3"])

    out = {
        "theta": theta,
        "phi": phi,
        "gD": gD,
        "t": t,
        "init_mode": init_mode,
        "psi0": psi0,
        "psi_t": psi_t,
        "rho_ABC": rho_ABC,
        "J_AB": J_AB,
        "J_BC": J_BC,
        "J_AC": J_AC,
        "J_CD": J_CD,
        **obs_state,
        **obs_probe,
    }

    if verbose:
        print(f"\n=== theta={theta:.3f}, phi={phi:.3f}, gD={gD:.2f}, t={t:.2f}, init={init_mode} ===")
        print("J_AB =", np.round(J_AB, 4))
        print("J_BC =", np.round(J_BC, 4))
        print("J_AC =", np.round(J_AC, 4))
        print("J_CD =", np.round(J_CD, 4))
        print("Top eigvals(rho_ABC) =", np.round(out["eigvals"][:4], 6))
        print(f"S(rho_ABC) = {out['S_ABC']:.6f}")
        print(f"I3 = {out['I3']:.6f}, I3_minus = {out['I3_minus']:.6f}")
        print(f"I(A:B) = {out['I_AB']:.6f}, I(A:C) = {out['I_AC']:.6f}, I(B:C) = {out['I_BC']:.6f}, P_pair = {out['P_pair']:.6f}")
        print(f"j_AB = {out['j_AB']:.6f}, j_BC = {out['j_BC']:.6f}, j_AC = {out['j_AC']:.6f}, J_tot = {out['J_tot']:.6f}")
        print(f"m_AB = {out['m_AB']:.6f}, m_BC = {out['m_BC']:.6f}, m_AC = {out['m_AC']:.6f}, J_rho_exp = {out['J_rho_exp']:.6f}, eta = {out['eta']:.6f}")

    return out

In [10]:
'''
Cell 10：Scan function
'''
def scan_fixed_norm(
    theta_vals,
    phi_vals,
    gD=0.20,
    t=0.10,
    init_mode="plus",
    rng_seed=None,
    custom_state=None,
    verbose_every=0,
):
    results = []
    counter = 0
    rng = np.random.default_rng(rng_seed)

    for th in theta_vals:
        for ph in phi_vals:
            counter += 1
            verbose = (verbose_every > 0 and counter % verbose_every == 0)

            try:
                res = run_fixed_norm_case(
                    th,
                    ph,
                    gD=gD,
                    t=t,
                    init_mode=init_mode,
                    rng=rng,
                    custom_state=custom_state,
                    verbose=verbose,
                )
                results.append(res)
            except ValueError:
                continue

    return results

In [11]:
'''
Cell 11：Summary helpers
'''
def summarize_scan_numeric(results):
    if len(results) == 0:
        return {
            "N": 0,
            "corr_J_rho_exp": np.nan,
            "corr_J_tot": np.nan,
            "corr_eta": np.nan,
            "corr_P_pair": np.nan,
            "max_I3_minus": np.nan,
            "mean_I3_minus": np.nan,
        }

    I3m = np.array([r["I3_minus"] for r in results], dtype=float)
    Jr  = np.array([r["J_rho_exp"] for r in results], dtype=float)
    Jt  = np.array([r["J_tot"] for r in results], dtype=float)
    Eta = np.array([r["eta"] for r in results], dtype=float)
    Pp  = np.array([r["P_pair"] for r in results], dtype=float)

    return {
        "N": len(results),
        "corr_J_rho_exp": safe_corr(Jr, I3m),
        "corr_J_tot": safe_corr(Jt, I3m),
        "corr_eta": safe_corr(Eta, I3m),
        "corr_P_pair": safe_corr(Pp, I3m),
        "max_I3_minus": float(np.max(I3m)),
        "mean_I3_minus": float(np.mean(I3m)),
    }

def summarize_scan(results, top_k=10):
    if len(results) == 0:
        print("No results.")
        return

    I3m = np.array([r["I3_minus"] for r in results], dtype=float)
    Jr  = np.array([r["J_rho_exp"] for r in results], dtype=float)
    Jt  = np.array([r["J_tot"] for r in results], dtype=float)
    Eta = np.array([r["eta"] for r in results], dtype=float)
    Pp  = np.array([r["P_pair"] for r in results], dtype=float)

    corr_Jr_I3m  = safe_corr(Jr, I3m)
    corr_Jt_I3m  = safe_corr(Jt, I3m)
    corr_Eta_I3m = safe_corr(Eta, I3m)
    corr_Pp_I3m  = safe_corr(Pp, I3m)

    print("\n=== Scan summary ===")
    print(f"N = {len(results)}")
    print(f"Pearson corr(J_rho_exp, I3_minus) = {corr_Jr_I3m:.6f}")
    print(f"Pearson corr(J_tot,     I3_minus) = {corr_Jt_I3m:.6f}")
    print(f"Pearson corr(eta,       I3_minus) = {corr_Eta_I3m:.6f}")
    print(f"Pearson corr(P_pair,    I3_minus) = {corr_Pp_I3m:.6f}")

    idx = np.argsort(-I3m)[:top_k]
    print(f"\nTop {top_k} points by I3_minus:")
    for k in idx:
        r = results[k]
        print(
            f"theta={r['theta']:.3f}, phi={r['phi']:.3f}, "
            f"I3_minus={r['I3_minus']:.6f}, "
            f"J_rho_exp={r['J_rho_exp']:.6f}, "
            f"J_tot={r['J_tot']:.6f}, "
            f"eta={r['eta']:.6f}, "
            f"P_pair={r['P_pair']:.6f}"
        )

def scan_over_time(
    theta_vals,
    phi_vals,
    t_vals,
    gD=0.20,
    init_mode="plus",
    rng_seed=None,
    custom_state=None,
):
    all_summaries = []
    for t in t_vals:
        results_t = scan_fixed_norm(
            theta_vals,
            phi_vals,
            gD=gD,
            t=t,
            init_mode=init_mode,
            rng_seed=rng_seed,
            custom_state=custom_state,
        )
        summary_t = summarize_scan_numeric(results_t)
        summary_t["t"] = t
        all_summaries.append(summary_t)
    return all_summaries

In [12]:
'''
Cell 12：Parameter presets
'''
theta_vals = np.linspace(0.1, np.pi/2, 20)
phi_vals   = np.linspace(0.0, np.pi/2, 20)

gD_default = 0.20
t_default  = 0.10

In [13]:
'''
Cell 13：Fixed initial states for reproducible experiments
'''
# Fixed random-product initial state
rng_rp = np.random.default_rng(42)
psi0_random_product_fixed = make_initial_state(init_mode="random_product", rng=rng_rp)

# Fixed Haar 4-qubit initial state
rng_haar = np.random.default_rng(42)
psi0_haar_fixed = make_initial_state(init_mode="haar", rng=rng_haar)

# Plus state is generated internally, but we keep a copy here for reference
psi0_plus = make_initial_state(init_mode="plus")

In [14]:
'''
Cell 14：Single-point sanity check
'''
res_test = run_fixed_norm_case(
    theta=0.7,
    phi=0.8,
    gD=gD_default,
    t=t_default,
    init_mode="plus",
    verbose=True,
)


=== theta=0.700, phi=0.800, gD=0.20, t=0.10, init=plus ===
J_AB = [1.2979 0.9355 0.6635]
J_BC = [0.8896 1.145  0.9475]
J_AC = [0.5608 0.385  1.5929]
J_CD = [0.2099 0.1729 0.2146]
Top eigvals(rho_ABC) = [9.997e-01 3.000e-04 0.000e+00 0.000e+00]
S(rho_ABC) = 0.003938
I3 = 0.000081, I3_minus = 0.000000
I(A:B) = 0.015710, I(A:C) = 0.204657, I(B:C) = 0.011057, P_pair = 0.231424
j_AB = 11.344133, j_BC = 12.302455, j_AC = 11.179250, J_tot = 20.124994
m_AB = -0.380825, m_BC = -0.201122, m_AC = 0.179703, J_rho_exp = 0.466659, eta = 0.023188


In [15]:
'''
Cell 15：Main experiment A — |+++⟩⊗|0⟩ short-time scan
'''
results_plus_t01 = scan_fixed_norm(
    theta_vals,
    phi_vals,
    gD=gD_default,
    t=t_default,
    init_mode="plus",
)

summarize_scan(results_plus_t01, top_k=8)


=== Scan summary ===
N = 400
Pearson corr(J_rho_exp, I3_minus) = 0.808442
Pearson corr(J_tot,     I3_minus) = 0.021477
Pearson corr(eta,       I3_minus) = 0.774692
Pearson corr(P_pair,    I3_minus) = 0.217473

Top 8 points by I3_minus:
theta=1.261, phi=1.571, I3_minus=0.003507, J_rho_exp=2.793624, J_tot=14.363786, eta=0.194491, P_pair=0.537814
theta=1.184, phi=1.571, I3_minus=0.003487, J_rho_exp=3.022546, J_tot=16.651461, eta=0.181518, P_pair=0.486946
theta=1.261, phi=1.488, I3_minus=0.003430, J_rho_exp=2.809099, J_tot=14.546059, eta=0.193118, P_pair=0.534703
theta=1.184, phi=1.488, I3_minus=0.003422, J_rho_exp=3.036369, J_tot=16.711357, eta=0.181695, P_pair=0.483793
theta=1.184, phi=1.405, I3_minus=0.003270, J_rho_exp=2.992101, J_tot=17.006858, eta=0.175935, P_pair=0.472623
theta=1.261, phi=1.405, I3_minus=0.003270, J_rho_exp=2.769735, J_tot=15.224359, eta=0.181928, P_pair=0.522633
theta=1.339, phi=1.571, I3_minus=0.003233, J_rho_exp=2.389832, J_tot=11.523562, eta=0.207387, P_pair=0.

In [16]:
'''
Cell 16：Main experiment B — fixed random-product initial state
'''
results_rp_fixed_t01 = scan_fixed_norm(
    theta_vals,
    phi_vals,
    gD=gD_default,
    t=t_default,
    init_mode="custom",
    custom_state=psi0_random_product_fixed,
)

summarize_scan(results_rp_fixed_t01, top_k=8)


=== Scan summary ===
N = 400
Pearson corr(J_rho_exp, I3_minus) = 0.382886
Pearson corr(J_tot,     I3_minus) = 0.079847
Pearson corr(eta,       I3_minus) = 0.424907
Pearson corr(P_pair,    I3_minus) = -0.685991

Top 8 points by I3_minus:
theta=1.261, phi=0.992, I3_minus=0.001915, J_rho_exp=1.474184, J_tot=21.862061, eta=0.067431, P_pair=0.226706
theta=1.261, phi=0.909, I3_minus=0.001905, J_rho_exp=1.631689, J_tot=22.383821, eta=0.072896, P_pair=0.217338
theta=1.339, phi=1.075, I3_minus=0.001884, J_rho_exp=1.246268, J_tot=21.401521, eta=0.058233, P_pair=0.259436
theta=1.339, phi=1.157, I3_minus=0.001856, J_rho_exp=1.105127, J_tot=19.610705, eta=0.056353, P_pair=0.267426
theta=1.261, phi=1.075, I3_minus=0.001844, J_rho_exp=1.306284, J_tot=20.870975, eta=0.062589, P_pair=0.238097
theta=1.261, phi=0.827, I3_minus=0.001843, J_rho_exp=1.775707, J_tot=22.404506, eta=0.079257, P_pair=0.208976
theta=1.184, phi=0.744, I3_minus=0.001837, J_rho_exp=2.050096, J_tot=21.254539, eta=0.096454, P_pair=0

In [17]:
'''
Cell 17：Time scan for |+++⟩⊗|0⟩
'''
t_vals = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.00]

time_summaries_plus = scan_over_time(
    theta_vals,
    phi_vals,
    t_vals,
    gD=gD_default,
    init_mode="plus",
)

for row in time_summaries_plus:
    print(
        f"t={row['t']:.2f} | "
        f"corr(J_rho_exp, I3-)={row['corr_J_rho_exp']:.4f} | "
        f"corr(J_tot, I3-)={row['corr_J_tot']:.4f} | "
        f"corr(eta, I3-)={row['corr_eta']:.4f} | "
        f"corr(P_pair, I3-)={row['corr_P_pair']:.4f} | "
        f"max(I3-)={row['max_I3_minus']:.4f}"
    )

t=0.05 | corr(J_rho_exp, I3-)=0.8027 | corr(J_tot, I3-)=0.0191 | corr(eta, I3-)=0.7612 | corr(P_pair, I3-)=0.2056 | max(I3-)=0.0008
t=0.10 | corr(J_rho_exp, I3-)=0.8084 | corr(J_tot, I3-)=0.0215 | corr(eta, I3-)=0.7747 | corr(P_pair, I3-)=0.2175 | max(I3-)=0.0035
t=0.15 | corr(J_rho_exp, I3-)=0.7764 | corr(J_tot, I3-)=0.0202 | corr(eta, I3-)=0.7733 | corr(P_pair, I3-)=0.2333 | max(I3-)=0.0089
t=0.20 | corr(J_rho_exp, I3-)=0.6689 | corr(J_tot, I3-)=0.0134 | corr(eta, I3-)=0.7177 | corr(P_pair, I3-)=0.2505 | max(I3-)=0.0183
t=0.30 | corr(J_rho_exp, I3-)=0.3458 | corr(J_tot, I3-)=-0.0096 | corr(eta, I3-)=0.3925 | corr(P_pair, I3-)=0.2774 | max(I3-)=0.0556
t=0.50 | corr(J_rho_exp, I3-)=0.2593 | corr(J_tot, I3-)=-0.0063 | corr(eta, I3-)=0.2159 | corr(P_pair, I3-)=0.2767 | max(I3-)=0.1913
t=1.00 | corr(J_rho_exp, I3-)=0.0966 | corr(J_tot, I3-)=0.5571 | corr(eta, I3-)=-0.0609 | corr(P_pair, I3-)=0.5055 | max(I3-)=0.1997


In [18]:
'''
Cell 18：Time scan for fixed random-product initial state
'''
time_summaries_rp_fixed = scan_over_time(
    theta_vals,
    phi_vals,
    t_vals,
    gD=gD_default,
    init_mode="custom",
    custom_state=psi0_random_product_fixed,
)

for row in time_summaries_rp_fixed:
    print(
        f"t={row['t']:.2f} | "
        f"corr(J_rho_exp, I3-)={row['corr_J_rho_exp']:.4f} | "
        f"corr(J_tot, I3-)={row['corr_J_tot']:.4f} | "
        f"corr(eta, I3-)={row['corr_eta']:.4f} | "
        f"corr(P_pair, I3-)={row['corr_P_pair']:.4f} | "
        f"max(I3-)={row['max_I3_minus']:.4f}"
    )

t=0.05 | corr(J_rho_exp, I3-)=0.4328 | corr(J_tot, I3-)=0.1899 | corr(eta, I3-)=0.4075 | corr(P_pair, I3-)=-0.5755 | max(I3-)=0.0005
t=0.10 | corr(J_rho_exp, I3-)=0.3829 | corr(J_tot, I3-)=0.0798 | corr(eta, I3-)=0.4249 | corr(P_pair, I3-)=-0.6860 | max(I3-)=0.0019
t=0.15 | corr(J_rho_exp, I3-)=0.4569 | corr(J_tot, I3-)=-0.0464 | corr(eta, I3-)=0.5475 | corr(P_pair, I3-)=-0.6867 | max(I3-)=0.0039
t=0.20 | corr(J_rho_exp, I3-)=0.5624 | corr(J_tot, I3-)=-0.1283 | corr(eta, I3-)=0.6677 | corr(P_pair, I3-)=-0.6011 | max(I3-)=0.0073
t=0.30 | corr(J_rho_exp, I3-)=0.5492 | corr(J_tot, I3-)=-0.0461 | corr(eta, I3-)=0.5923 | corr(P_pair, I3-)=-0.5049 | max(I3-)=0.0208
t=0.50 | corr(J_rho_exp, I3-)=0.2018 | corr(J_tot, I3-)=0.3955 | corr(eta, I3-)=0.0497 | corr(P_pair, I3-)=-0.3039 | max(I3-)=0.0830
t=1.00 | corr(J_rho_exp, I3-)=0.5552 | corr(J_tot, I3-)=0.5106 | corr(eta, I3-)=0.3981 | corr(P_pair, I3-)=0.4199 | max(I3-)=0.2660


In [19]:
'''
Cell 19：Optional — fixed Haar initial state scan
'''
results_haar_fixed_t01 = scan_fixed_norm(
    theta_vals,
    phi_vals,
    gD=gD_default,
    t=t_default,
    init_mode="custom",
    custom_state=psi0_haar_fixed,
)

summarize_scan(results_haar_fixed_t01, top_k=8)


=== Scan summary ===
N = 400
Pearson corr(J_rho_exp, I3_minus) = -0.091665
Pearson corr(J_tot,     I3_minus) = 0.084456
Pearson corr(eta,       I3_minus) = -0.180550
Pearson corr(P_pair,    I3_minus) = -0.641887

Top 8 points by I3_minus:
theta=1.571, phi=1.157, I3_minus=0.698101, J_rho_exp=1.739839, J_tot=20.063373, eta=0.086717, P_pair=1.139625
theta=1.571, phi=1.240, I3_minus=0.698063, J_rho_exp=1.507859, J_tot=16.773089, eta=0.089897, P_pair=1.137729
theta=1.571, phi=1.075, I3_minus=0.696102, J_rho_exp=1.885885, J_tot=22.547061, eta=0.083642, P_pair=1.141907
theta=1.571, phi=1.323, I3_minus=0.695637, J_rho_exp=1.195196, J_tot=12.831163, eta=0.093148, P_pair=1.136013
theta=1.571, phi=0.992, I3_minus=0.692526, J_rho_exp=1.949155, J_tot=24.159642, eta=0.080678, P_pair=1.144669
theta=0.255, phi=0.000, I3_minus=0.692004, J_rho_exp=1.171109, J_tot=17.204892, eta=0.068068, P_pair=1.099491
theta=1.571, phi=1.405, I3_minus=0.690713, J_rho_exp=0.818553, J_tot=8.491443, eta=0.096397, P_pair=